# Q1 — Decision Tree using Hunt’s Algorithm (Manual Implementation)

This notebook builds a decision tree **without using sklearn**, implementing the logic of **Hunt’s algorithm** manually using **entropy** and **information gain**.

### Hunt’s Algorithm Steps:
1. If all samples belong to one class → make a **leaf node**.
2. If there are mixed classes → compute **information gain** for each attribute.
3. Choose the attribute with **highest gain** and **split** the dataset.
4. **Recursively** apply the above steps to each subset.
5. Stop when all nodes are pure or no more attributes remain.

In [1]:
import pandas as pd
import math
from pprint import pprint

data = {
    'Home Owner': ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'Marital Status': ['Single','Married','Single','Married','Divorced','Married','Divorced','Single','Married','Single'],
    'Annual Income': [125,100,70,120,95,60,220,85,75,90],
    'Defaulted Borrower': ['No','No','No','No','Yes','Yes','No','Yes','No','Yes']
}

df = pd.DataFrame(data)
df

,Home Owner,Marital Status,Annual Income,Defaulted Borrower
0,Yes,Single,125,No
1,No,Married,100,No
2,No,Single,70,No
3,Yes,Married,120,No
4,No,Divorced,95,Yes
5,No,Married,60,Yes
6,Yes,Divorced,220,No
7,No,Single,85,Yes
8,No,Married,75,No
9,No,Single,90,Yes


In [2]:
def entropy(target_col):
    elements, counts = zip(*target_col.value_counts().items())
    entropy_val = 0
    for i in range(len(elements)):
        p = counts[i]/sum(counts)
        entropy_val += -p*math.log2(p)
    return entropy_val

def info_gain(data, split_attr, target_attr):
    total_entropy = entropy(data[target_attr])
    vals, counts = zip(*data[split_attr].value_counts().items())
    weighted_entropy = 0
    for i in range(len(vals)):
        subset = data[data[split_attr] == vals[i]]
        weighted_entropy += (counts[i]/sum(counts)) * entropy(subset[target_attr])
    gain = total_entropy - weighted_entropy
    return gain

In [3]:
# Recursive Hunt’s algorithm implementation
def hunt_tree(data, features, target='Defaulted Borrower'):
    # Base cases
    if len(data[target].unique()) == 1:
        return data[target].iloc[0]
    if not features:
        return data[target].mode()[0]

    # Compute best attribute based on information gain
    gains = {feat: info_gain(data, feat, target) for feat in features}
    best_feat = max(gains, key=gains.get)

    tree = {best_feat: {}}
    for val in data[best_feat].unique():
        subset = data[data[best_feat] == val]
        remaining_feats = [f for f in features if f != best_feat]
        tree[best_feat][val] = hunt_tree(subset, remaining_feats, target)
    return tree

In [4]:
# Build decision tree
features = ['Home Owner', 'Marital Status']
tree = hunt_tree(df, features)
print("Decision Tree:")
pprint(tree)

Decision Tree:
{'Home Owner': {'No': {'Marital Status': {'Divorced': 'Yes',
                                          'Married': 'No',
                                          'Single': 'Yes'}},
                'Yes': 'No'}}


In [6]:
# Add simple prediction function
def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree
    attr = next(iter(tree))
    val = sample.get(attr)
    if val not in tree[attr]:
        return 'Unknown'
    return predict(tree[attr][val], sample)

# Example test
sample = {'Home Owner': 'No', 'Marital Status': 'Divorced'}
print("Prediction for", sample, "→", predict(tree, sample))

Prediction for {'Home Owner': 'No', 'Marital Status': 'Divorced'} → Yes


### ✅ Explanation
- The algorithm recursively splits based on **maximum information gain**.
- If a subset is pure (all same class), a **leaf node** is created.
- The final tree is a nested dictionary showing the decisions.
- The `predict()` function traverses this dictionary for new samples.